# SpleenGLASS: HD95 on the 60 v2 checkpoints (inference only, no training)

**Before running:** GPU = T4, Internet ON, attach these datasets: `spleenglass-preprocessed-cache` and all five `spleenglass-v2-ckpt-*`
(abdulrehmancs24, alihassansiddiqui80, ahmadcs24, nadirjamal38, forabd). Do NOT attach the old `spleenglass-ckpt-alihassansiddiqui80` (v1).

**Cell 1: setup.** Clones the repo and checks out commit `485d1dc` (the commit all 60 runs used), copies the preprocessed cache, downloads and SHA-checks the SAM checkpoint, and prints the end of `src/metrics.py` so you can verify its provenance.

In [ ]:
import os, sys, json, glob, shutil, subprocess, hashlib, time, collections
import numpy as np, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from kaggle_secrets import UserSecretsClient

TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_DIR = "/kaggle/working/spleenglass_pilot_run"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
r = subprocess.run(["git", "clone", f"https://{TOKEN}@github.com/abdul-rehman-24/spleenglass_pilot_run.git", REPO_DIR],
                   capture_output=True, text=True)
assert r.returncode == 0, "git clone failed (token or network)"
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
r = subprocess.run(["git", "checkout", "485d1dc"], capture_output=True, text=True)
assert r.returncode == 0, r.stderr
print("repo commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

# preprocessed cache -> where VolumeCache expects it
os.makedirs("/kaggle/working/cache/preprocessed", exist_ok=True)
hits = [p for p in glob.glob("/kaggle/input/**/spleenglass-preprocessed-cache", recursive=True) if os.path.isdir(p)]
if not hits:
    hits = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/*.npz", recursive=True)})
assert hits, "preprocessed-cache dataset is not attached"
shutil.copytree(hits[0], "/kaggle/working/cache/preprocessed", dirs_exist_ok=True)
print("cache files:", len(glob.glob("/kaggle/working/cache/preprocessed/*.npz")), "(expect 41)")

# SAM base checkpoint (needed to build SAMed before loading the trained weights)
os.system("pip install git+https://github.com/facebookresearch/segment-anything.git -q")
SAM_CKPT_PATH = "/kaggle/working/sam_vit_b_01ec64.pth"
if not os.path.exists(SAM_CKPT_PATH):
    os.system(f"wget -q -O {SAM_CKPT_PATH} https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth")
sha = hashlib.sha256(open(SAM_CKPT_PATH, "rb").read()).hexdigest()
assert sha == "ec2df62732614e57411cdcf32a23ffdf28910380d03139ee0f4fcbe91eb8c912", "SAM checkpoint SHA mismatch"
print("SAM checkpoint SHA OK")

# provenance check: compare with the metrics.py you pasted
print("--- tail of src/metrics.py at this commit ---")
print("".join(open("src/metrics.py").readlines()[-6:]))
print("md5 metrics.py:", hashlib.md5(open("src/metrics.py", "rb").read()).hexdigest())

**Cell 2: check the voxel spacing.** HD95 is in millimetres, so it needs the spacing of the cached arrays. The protocol says 1.5 x 1.5 x 2.0 mm with only a centre crop/pad in-plane (no resizing). This cell prints what the cache actually stores. **Send me this output.**

In [ ]:
d = np.load("/kaggle/working/cache/preprocessed/spleen_9.npz")
print("npz keys:", d.files)
for k in d.files:
    print(k, d[k].shape, d[k].dtype)
for k in d.files:
    if "spac" in k.lower() or "affine" in k.lower():
        print(k, "=", d[k])
import pandas as pd
mp = "logs/preprocessing_metadata.csv"
if os.path.exists(mp):
    m = pd.read_csv(mp); print(m.columns.tolist()); print(m.head(3).to_string())

**Cell 3: HD95 functions (tested).** Surface voxels = mask voxels touching background (6-neighbourhood). Distances in mm via Euclidean distance transform with the voxel spacing. `hd95_max` = max of the two directed 95th percentiles (MONAI convention, the one to report); `hd95_pooled` = 95th percentile of both directions pooled (medpy style), saved only as a sensitivity check. An empty prediction gives NaN. I tested this on synthetic cases: identical masks give 0, translated boxes give exactly shift x spacing, and it matches a brute-force implementation to 1e-6.

In [ ]:
from scipy import ndimage as ndi
SPACING = (1.5, 1.5, 2.0)   # mm, array axes (H, W, S). Confirm with Cell 2 before trusting HD95.
_STRUCT = ndi.generate_binary_structure(3, 1)

def surface_mask(mask):
    m = mask.astype(bool)
    return m & ~ndi.binary_erosion(m, structure=_STRUCT, border_value=0)

def hd_pair(pred, gt, spacing=SPACING, percentile=95.0):
    sp, sg = surface_mask(pred), surface_mask(gt)
    if sp.sum() == 0 or sg.sum() == 0:
        return float("nan"), float("nan")
    d1 = ndi.distance_transform_edt(~sg, sampling=spacing)[sp]   # pred surface -> gt surface
    d2 = ndi.distance_transform_edt(~sp, sampling=spacing)[sg]   # gt surface -> pred surface
    hd_max = float(max(np.percentile(d1, percentile), np.percentile(d2, percentile)))
    hd_pooled = float(np.percentile(np.concatenate([d1, d2]), percentile))
    return hd_max, hd_pooled

# quick self-test (should print 0.0 and 8.0)
a = np.zeros((40, 40, 30), bool); a[10:30, 10:30, 8:22] = True
print(hd_pair(a, a)[0], hd_pair(a, np.roll(a, 4, axis=2), percentile=100)[0])

**Cell 4: model loading and per-checkpoint evaluation.** For one checkpoint: rebuild the model, load the saved best weights, predict the 9 validation volumes with exactly the same steps as `validate_3d` (32-slice batches, sigmoid > 0.5), then compute DSC, HD95 and voxel counts. The recomputed DSC must match the DSC stored in the checkpoint (tolerance 1e-3); otherwise the row is flagged.

In [ ]:
from src.datasets import VolumeCache
from src.metrics import dice_score_3d
from src.models.samed import build_small_sam, LoRA_Sam
from src.models.attention_unet import AttentionUNet

device = torch.device("cuda")
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)
volume_cache = VolumeCache()

def make_samed_model():
    sam, _ = build_small_sam(SAM_CKPT_PATH)
    return LoRA_Sam(sam, r=4)

CTOR = {"attentionunet": AttentionUNet, "samed": make_samed_model}
CKPT = {os.path.basename(p): p for p in glob.glob("/kaggle/input/**/v2_N*_best.pt", recursive=True)}
assert len(CKPT) == 60, len(CKPT)

@torch.no_grad()
def predict_volume(model, img_3d, lbl_3d, batch_size=32):
    S = img_3d.shape[-1]
    preds = np.zeros_like(lbl_3d)
    for start in range(0, S, batch_size):
        end = min(start + batch_size, S)
        batch = np.transpose(img_3d[:, :, start:end], (2, 0, 1))
        x = torch.from_numpy(batch).unsqueeze(1).to(device)
        probs = torch.sigmoid(model(x)).squeeze(1).cpu().numpy()
        preds[:, :, start:end] = (np.transpose(probs, (1, 2, 0)) > 0.5).astype(np.float32)
    return preds

def evaluate_checkpoint(run_id):
    ck = torch.load(CKPT[f"{run_id}_best.pt"], map_location="cpu", weights_only=False)
    assert ck["run_id"] == run_id
    kind = "samed" if "_samed_" in run_id else "attentionunet"
    model = CTOR[kind]()
    model.load_state_dict(ck["model_state_dict"])
    model = model.to(device).eval()
    rows = []
    for i, vid in enumerate(ck["validation_ids"]):
        img, lbl = volume_cache.load(vid)
        pred = predict_volume(model, img, lbl)
        dsc = float(dice_score_3d(pred, lbl))
        hd_max, hd_pool = hd_pair(pred > 0.5, lbl > 0.5)
        logged = ck["per_volume_val_dsc"][i]
        rows.append({"run_id": run_id, "volume_id": vid, "dsc_recomputed": dsc, "dsc_logged": logged,
                     "dsc_match": abs(dsc - logged) < 1e-3, "hd95_mm": hd_max, "hd95_pooled_mm": hd_pool,
                     "pred_voxels": int((pred > 0.5).sum()), "gt_voxels": int((lbl > 0.5).sum()),
                     "best_epoch": ck["best_epoch"]})
    del model, ck
    torch.cuda.empty_cache()
    return rows

**Cell 5: smoke test on 2 checkpoints (one per model).** Check that `dsc_match` is True for all 9 volumes and the HD95 values look sensible (a few mm for good segmentations, tens of mm for failures). **Send me this output before running the full loop.**

In [ ]:
import pandas as pd
t0 = time.time()
test = []
for rid in ["v2_N20_random_attentionunet_seed42", "v2_N05_random_samed_seed42"]:
    test += evaluate_checkpoint(rid)
    print(rid, "done", round(time.time() - t0), "s")
df_t = pd.DataFrame(test)
print(df_t.drop(columns=["run_id"]).round(4).to_string())
print("ALL DSC MATCH:", df_t.dsc_match.all(), "| max |diff|:", (df_t.dsc_recomputed - df_t.dsc_logged).abs().max())

**Cell 6: full run over all 60 checkpoints.** Resume-safe: results are appended to `/kaggle/working/hd95_v2.csv` after every checkpoint, and finished run_ids are skipped if the notebook is restarted.

In [ ]:
OUT = "/kaggle/working/hd95_v2.csv"
done = set(pd.read_csv(OUT).run_id) if os.path.exists(OUT) else set()
run_ids = sorted(n[:-len("_best.pt")] for n in CKPT)
t0 = time.time()
for k, rid in enumerate(run_ids, 1):
    if rid in done:
        continue
    rows = evaluate_checkpoint(rid)
    pd.DataFrame(rows).to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)
    bad = [r["volume_id"] for r in rows if not r["dsc_match"]]
    print(f"[{k}/60] {rid} | mean HD95 {np.nanmean([r['hd95_mm'] for r in rows]):.2f} mm"
          f"{' | DSC MISMATCH: ' + str(bad) if bad else ''} | {time.time() - t0:.0f}s")
print("finished")

**Cell 7: final sanity check and summary.** Expect 540 rows (60 runs x 9 volumes), all DSC matches True, and a count of empty predictions (NaN HD95). Then the mean HD95 per model, budget and strategy. Download `hd95_v2.csv` from the Output tab and send it to me.

In [ ]:
h = pd.read_csv(OUT)
print("rows:", len(h), "| runs:", h.run_id.nunique(), "| all DSC match:", h.dsc_match.all(),
      "| NaN HD95 (empty pred):", int(h.hd95_mm.isna().sum()))
h["model"] = np.where(h.run_id.str.contains("_samed_"), "samed", "attentionunet")
h["N"] = h.run_id.str.extract(r"v2_N(\d+)_")[0].astype(int)
h["strategy"] = h.run_id.str.extract(r"v2_N\d+_([a-z]+)_")[0]
print(h[h.N < 32].groupby(["model", "N", "strategy"]).hd95_mm.agg(["mean", "median", "max"]).round(2).to_string())